# 23-12 · Строим план из списка файлов

Практика к разделу [«От анализа к плану действий»](../../site/chapters/glava-23/23-11-plan-dejstvij.html). Настоящий файл — `projects/python/safesort/src/safesort/planner.py`.

## Цель

Воспроизвести `build_plan()` и убедиться, что каждый файл получает предсказуемый путь назначения `root/Sorted/<категория>/<имя>` — как обычные данные, без единого изменения файловой системы.

## Example — модели и вспомогательные функции

In [ ]:
from dataclasses import dataclass
from pathlib import Path

OTHER_CATEGORY = "other"
DEFAULT_EXTENSIONS = {
    "documents": [".pdf", ".docx", ".txt", ".odt"],
    "images": [".jpg", ".jpeg", ".png", ".webp"],
    "archives": [".zip", ".tar", ".gz", ".7z"],
}


def classify(extension, mapping):
    normalized = extension.lower()
    for category, extensions in mapping.items():
        if normalized in {ext.lower() for ext in extensions}:
            return category
    return OTHER_CATEGORY


@dataclass(frozen=True)
class FileInfo:
    path: Path
    size: int
    extension: str


@dataclass(frozen=True)
class MoveOperation:
    source: Path
    destination: Path


@dataclass(frozen=True)
class SortPlan:
    root: Path
    operations: tuple

## build_plan() — план как данные

In [ ]:
def _resolve_collision(candidate, reserved):
    if candidate not in reserved and not candidate.exists():
        return candidate
    stem, suffix, parent = candidate.stem, candidate.suffix, candidate.parent
    counter = 1
    while True:
        alternative = parent / f"{stem} ({counter}){suffix}"
        if alternative not in reserved and not alternative.exists():
            return alternative
        counter += 1


def build_plan(files, root, destination_name, extensions_mapping):
    root = Path(root)
    dest_root = root / destination_name
    reserved = set()
    operations = []
    for file in files:
        category = classify(file.extension, extensions_mapping)
        dest_dir = dest_root / category
        candidate = dest_dir / file.path.name
        destination = _resolve_collision(candidate, reserved)
        reserved.add(destination)
        operations.append(MoveOperation(source=file.path, destination=destination))
    return SortPlan(root=root, operations=tuple(operations))


koren = Path("/home/anna/Downloads")
fajly = [
    FileInfo(path=koren / "otchet.pdf", size=1200, extension=".pdf"),
    FileInfo(path=koren / "photo.jpg", size=204800, extension=".jpg"),
    FileInfo(path=koren / "archiv.zip", size=5000, extension=".zip"),
]

plan = build_plan(fajly, koren, "Sorted", DEFAULT_EXTENSIONS)
for op in plan.operations:
    print(op.source, "->", op.destination)

## Проверка результата

In [ ]:
destinations = {op.source.name: op.destination for op in plan.operations}
assert destinations["otchet.pdf"] == koren / "Sorted" / "documents" / "otchet.pdf"
assert destinations["photo.jpg"] == koren / "Sorted" / "images" / "photo.jpg"
assert destinations["archiv.zip"] == koren / "Sorted" / "archives" / "archiv.zip"
print("Верно: каждый файл получил путь Sorted/<категория>/<имя>.")

## Starter

Заполните отмеченное место. Неизменённый starter не проходит tests.

In [ ]:
def plan_s_dopolnitelnym_fajlom(filename: str):
    # TODO: create FileInfo, append it to fajly, then call build_plan().
    raise NotImplementedError


plan2 = plan_s_dopolnitelnym_fajlom("strannyj.xyz")

## Task

Напишите функцию, добавляющую к исходным данным один файл и строящую для него plan.

## Tests

Запустите после task cell: есть основной пример и хотя бы один крайний случай.

In [ ]:
destination = next(op.destination for op in plan2.operations if op.source.name == "strannyj.xyz")
assert destination == koren / "Sorted" / "other" / "strannyj.xyz"
plan_pdf = plan_s_dopolnitelnym_fajlom("REPORT.PDF")
dest_pdf = next(op.destination for op in plan_pdf.operations if op.source.name == "REPORT.PDF")
assert dest_pdf.parent.name == "documents"
print("Tests passed")

## Hint

Расширение можно получить как `Path(filename).suffix`; для классификатора нормализация уже реализована.

## Solution

<details><summary>Показать решение после собственной попытки</summary>

```python
def plan_s_dopolnitelnym_fajlom(filename: str):
    path = koren / filename
    extra = FileInfo(path=path, size=10, extension=path.suffix)
    return build_plan(fajly + [extra], koren, "Sorted", DEFAULT_EXTENSIONS)


plan2 = plan_s_dopolnitelnym_fajlom("strannyj.xyz")
```

</details>